# Обучение модели (RAVDESS в `../datas`)

Датасет: `c:/projects/hibi/datas` — папки `Actor_01` … `Actor_24` с файлами `03-01-EMOTION-....wav`.

Эмоции RAVDESS → классы приложения:
- **calm**: neutral, calm
- **joyful**: happy, surprised
- **tense**: sad, angry, fearful, disgust

Быстрый запуск без ноутбука: `python ml/train_from_datas.py` из папки `hibiki1`.

In [ ]:
from pathlib import Path

# Путь к datas рядом с hibiki1
DATAS_DIR = Path("..") / "datas"
OUTPUT_DIR = Path("output")
ASSETS_DIR = Path("..") / "assets" / "models"
MODEL_NAME = "voice_mood"
LABELS = ["calm", "joyful", "tense"]

SAMPLE_RATE = 16000
DURATION_SEC = 3.0
N_MFCC = 20
N_FFT = 512
HOP_LENGTH = 256
N_MELS = 40

RAVDESS_EMOTION_TO_MOOD = {
    "01": "calm", "02": "calm", "03": "joyful", "04": "tense",
    "05": "tense", "06": "tense", "07": "tense", "08": "joyful",
}

In [ ]:
%pip install -q librosa soundfile scikit-learn tensorflow numpy

In [ ]:
import os
from pathlib import Path

import librosa
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

TARGET_SAMPLES = int(SAMPLE_RATE * DURATION_SEC)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)


def parse_mood(path: Path) -> str | None:
    parts = path.stem.split("-")
    if len(parts) < 3 or parts[0] != "03" or parts[1] != "01":
        return None
    return RAVDESS_EMOTION_TO_MOOD.get(parts[2])


def extract_features(path: Path) -> np.ndarray:
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    if len(y) > TARGET_SAMPLES:
        y = y[:TARGET_SAMPLES]
    elif len(y) < TARGET_SAMPLES:
        y = np.pad(y, (0, TARGET_SAMPLES - len(y)))
    mfcc = librosa.feature.mfcc(
        y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC, n_fft=N_FFT,
        hop_length=HOP_LENGTH, n_mels=N_MELS,
    )
    return mfcc.mean(axis=1).astype(np.float32)


X, y = [], []
files = list(DATAS_DIR.glob("Actor_*/*.wav"))
print(f"Found {len(files)} wav in {DATAS_DIR.resolve()}")

for fp in files:
    mood = parse_mood(fp)
    if mood is None:
        continue
    X.append(extract_features(fp))
    y.append(mood)

X = np.stack(X)
y = np.array(y)
print("Samples:", X.shape, "| calm:", (y=="calm").sum(), "joyful:", (y=="joyful").sum(), "tense:", (y=="tense").sum())

In [ ]:
le = LabelEncoder()
le.fit(LABELS)
y_enc = le.transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(N_MFCC,)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(len(LABELS), activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=25, batch_size=64)

pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print(classification_report(y_test, pred, target_names=le.classes_))

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()

tflite_out = ASSETS_DIR / f"{MODEL_NAME}.tflite"
labels_out = ASSETS_DIR / "labels.txt"
tflite_out.write_bytes(tflite_bytes)
labels_out.write_text("\n".join(le.classes_), encoding="utf-8")

print("Saved:", tflite_out.resolve())
print("Saved:", labels_out.resolve())
print("Добавьте в pubspec.yaml: assets/models/voice_mood.tflite")